## PCA:
A dimension reduction method, which projects the original high-dim data onto a lower dimensional plane while preserving as much variance as possible (depends on # of PCs selected).
The PCs refers some vectors that are orthogonal to each other, each denotes one axis, and the sign of a PC does not matter.
Before applying PCA, X should be centered (each entry should minus its column mean).

Codes below demonstrate two ways of applying PCA in python.

In [ ]:
# apply PCA manually
import numpy as np
from numpy import linalg as la
X_centered = X - X.mean(axis = 0)               # 0: 垂直方向，1: 水平方向, 这里就是对X做中心化
U, S, VT = la.svd(X_centered)
pc1 = VT.T[:, 0]
pc2 = VT.T[:, 1]
# ...
W = VT.T[:, 0:2]                                # 构建W矩阵，只取前两个PC
Xproj = X@W


# apply PCA using sklearn:
from sklearn.decomposition import PCA
pca = PCA(n_components = 2)                     # 只用前两个主成分
Xproj = pca.fit_transform(X)                    # PCA()会自动对X进行中心化

pca.explained_variance_ratio_                   # 展示每个主成分分别保留了多少variance

## How to select the right number of PCs:
method 1: by setting n_components to a value between 0.0 and 1.0, PCA() selects the smallest possible number of PCs such that the total variance preserved by those PCs is greater than the specified value.

method 2: display the graph of variance preserved vs the number of PCs selected. There will be an elbow at the point when variance preserved stops increasing significantly.

In [ ]:
# method 1:
from sklearn.decomposition import PCA
pca = PCA(n_components = 0.9)     
Xproj = pca.fit_transform(X)

# method 2:
import matplotlib.pyplot as plt

pca = PCA()
pca.fit(X)                                              # 训练一个不限维度的PCA
cumsum = np.cumsum(pca.explained_variance_ratio_)       # 计算累计方差解释率

plt.figure(figsize=(8, 5))
plt.plot(cumsum, linewidth=2.5)                         # 绘制累计方差曲线
d = np.argmax(cumsum >= 0.95) + 1                       # 假设我们想找出达到 95% 方差对应的最佳PC数量
plt.plot([d, d], [0, 0.95], "k:")                       # 竖直虚线
plt.plot([0, d], [0.95, 0.95], "k:")                    # 水平虚线
plt.plot(d, 0.95, "ko")                                 # 交叉点处的黑点
plt.xlabel("Dimensions", fontsize=12)
plt.ylabel("Explained Variance", fontsize=12)
plt.axis([0, len(cumsum), 0, 1.05])
plt.grid(True)
plt.show()
              

## Decompress PCA:
suppose the columns of W are the selected PCs and Xproj is the dim-reduced data. $X_{recovered} = X_{proj} W^{T}$.

In [ ]:
pca = PCA(n_components = 2)
Xproj = pca.fit_transform(X)
Xrecovered = pca.inverse_transform(Xproj)

## Randomized PCA:
can decrease computational complexity and give approximated PCs.

In [ ]:
random_pca = PCA(n_components = 2, svd_solver = 'random')
Xproj = random_pca.fit_transform(X)

## Incremental PCA:
renew the PCs step by step. Each time by inputing a mini-batch of the original data. can help save memory.

In [ ]:
from sklearn.decomposition import IncrementalPCA
n_batches = 100                                                 # 设定把原始数据集拆分成多少个mini batch
inc_pca = IncrementalPCA(n_components = 2)
for X_batch in np.array_split(X_train, n_batches):
    inc_pca.partial_fit(X_batch)
Xproj = inc_pca.transform(X_train)

## Kernel PCA:
kernel PCA extends PCA to nonlinear settings by mapping data into a high-dimensional feature space using kernel functions, allowing nonlinear structures to be linearized and unfolded.

In [ ]:
from sklearn.decomposition import KernelPCA
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

clf = Pipeline([                                        # pipeline中定义每一个参数组合的流程：先做kernel PCA，然后对reduced X做logistic regression。这里定义kernel PCA这一步叫做‘kpca’, logistic regression这一步叫做'log_reg'
    ('kpca', KernelPCA(n_components = 2)),
    ('log_reg', LogisticRegression())
])

param_grid = [{
    'kpca_kernel': ['rbf', 'sigmoid'],                  # parameter grid里的每一个要测试的参数，名字前面都得加pipeline里对应模型的名字，比如kernel是KernelPCA()中的参数，然后KernelPCA()在pipeline里命名为kpca, 那么kernel在parameter grid里就得写成kpca_kernel
    'kpca_gamma': np.linspace[0.01, 0.1, 10],
}]

grid_search = GridSearchCV(clf, param_grid, cv=3)
grid_search.fit(X, y)
grid_search.best_params_

kernel_pca = KernelPCA(n_components = 2, kernel="rbf", gamma=0.04)      # 重新运行KernelPCA()，这次用的参数是grid search输出的最佳参数组合
Xreduced = kernel_pca.fit_transform(X_train)

## LLE

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding
lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10)
X_reduced = lle.fit_transform(X)